In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name != "ecohome_solution":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"
VECTORSTORE_DIR = PROJECT_ROOT / "data" / "vectorstore"

print("Project:", PROJECT_ROOT)
print("Documents:", DOCUMENTS_DIR)
print("Vector store:", VECTORSTORE_DIR)

Project: c:\Users\praveen.vanaparthy\Downloads\New folder\ecohome_solution
Documents: c:\Users\praveen.vanaparthy\Downloads\New folder\ecohome_solution\data\documents
Vector store: c:\Users\praveen.vanaparthy\Downloads\New folder\ecohome_solution\data\vectorstore


In [2]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    str(DOCUMENTS_DIR),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documents = loader.load()

print(f"Documents loaded: {len(documents)}")

for doc in documents:
    print("-", Path(doc.metadata["source"]).name)

C:\Users\praveen.vanaparthy\AppData\Local\Temp\ipykernel_27092\3174300315.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Documents loaded: 6
- tip_energy_storage.txt
- tip_ev_optimization.txt
- tip_hvac_optimization.txt
- tip_renewable_energy.txt
- tip_seasonal_energy.txt
- tip_smart_home_automation.txt


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(documents)

print(f"Document chunks created: {len(chunks)}")

Document chunks created: 12


In [4]:
import os
from dotenv import load_dotenv
from langchain_cohere import CohereEmbeddings

load_dotenv()

if not os.getenv("COHERE_API_KEY"):
    raise ValueError("COHERE_API_KEY is not configured in .env")

embeddings = CohereEmbeddings(
    model="embed-english-v3.0"
)

print("Cohere embeddings initialized successfully.")

Cohere embeddings initialized successfully.


In [5]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
    collection_name="ecohome_energy_tips",
)

print("ChromaDB vector store created successfully.")

ChromaDB vector store created successfully.


In [6]:
query = "How can I reduce electricity cost when using my EV?"

results = vectorstore.similarity_search(query, k=3)

print(f"Retrieved results: {len(results)}")

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", Path(doc.metadata["source"]).name)
    print(doc.page_content[:500])

Retrieved results: 3

--- Result 1 ---
Source: tip_ev_optimization.txt
Electric Vehicle Charging Optimization Best Practices

EV charging is a flexible high-energy load and is therefore an important target for energy optimization.

When solar generation is available, schedule EV charging during strong solar production whenever the vehicle's required departure time allows.

If solar generation is unavailable, prefer lower electricity price periods such as off-peak hours.

EV scheduling should consider:
- Current battery state of charge
- Required charge level
- Vehi

--- Result 2 ---
Source: tip_ev_optimization.txt
Avoid charging unnecessarily during expensive peak periods.

For example, if an EV can charge overnight, compare overnight off-peak pricing with expected daytime solar generation. If the vehicle is available during the day and solar generation is strong, daytime solar charging may provide greater renewable-energy utilization.

The recommendation should always satisfy the requ

In [7]:
query = "How should I optimize my thermostat during expensive peak hours?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", Path(doc.metadata["source"]).name)
    print(doc.page_content[:400])


--- Result 1 ---
Source: tip_hvac_optimization.txt
Use weather forecasts when planning HVAC operation. On very hot afternoons, pre-cooling the home during lower-cost or high-solar periods can reduce expensive peak-hour consumption.

Keep doors and windows closed while heating or cooling is operating.

Use ceiling fans together with moderate thermostat settings to improve perceived comfort without significantly increasing electricity consumption.



--- Result 2 ---
Source: tip_hvac_optimization.txt
HVAC Energy Optimization Best Practices

HVAC systems are often one of the largest sources of residential electricity consumption.

Use programmable or smart thermostats to adjust temperatures automatically based on occupancy and electricity prices.

During periods of high electricity prices, slightly increase the cooling temperature or decrease heating intensity when comfort allows.

Avoid large 

--- Result 3 ---
Source: tip_seasonal_energy.txt
Seasonal Energy Management Best Practices

E